<a href="https://colab.research.google.com/github/MicheleQGF/ProjectAllocationBudgetSynthetic/blob/main/01_hraFusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Resource Allocation & Budget Analysis · Part 2 · Extract and clean the data and Part 3 · Team seniority and HRA matching


In [ ]:
import os
from pathlib import Path

# Este comando encuentra tu carpeta de usuario real, se llame como se llame
user_home = Path.home()
print("Tu carpeta de usuario real es:", user_home)


Tu carpeta de usuario real es: C:\Users\PC


In [ ]:
# Cambia la ruta por la carpeta que tú quieras
nueva_ruta = r"C:\Users\PC\Documents\TAWOS"

os.chdir(nueva_ruta)

# Verificamos que se haya cambiado
print("Ahora estás en:", os.getcwd())

Ahora estás en: C:\Users\PC\Documents\TAWOS


In [ ]:
import pandas as pd, numpy as np

date_cols = ['Creation_Date', 'Estimation_Date', 'Resolution_Date',
             'Last_Updated', 'Sprint_Start', 'Sprint_End']
main = pd.read_csv('tawos_data/main_part2.csv', parse_dates=date_cols)  # CSVs lose dtypes

## TAWOS Exploration
### Inferring Seniority Tiers from Jira Activity (TAWOS data)

**Goal:** The TAWOS Jira data has assignee IDs but no job levels. This notebook builds a **data-driven seniority tier** (Junior → Senior) for each assignee, using only how they behave in Jira. Those tiers can later be matched to the HRA (HR) profiles to assign hourly rates.

---

### 1. Per-assignee summary (`user_stats`)
Group all issues by `Assignee_ID` and compute:
- `Total_Issues`, `First_Activity`, `Last_Activity`
- `Avg_Story_Point`, `Avg_Resolution_Minutes`
- `Tenure_Days` = last activity − first activity

**What the data showed (271 assignees):** a long tail. Median is 4 issues, 25% have just 1 issue, while the max is 1,402.

### 2. Define the "core team"
Keep assignees with **≥ 10 issues and ≥ 30 days of tenure** → **97 of 271** people. These are the sustained contributors.

### 3. Detect true reassignments (`Change_Log`)
- Filter the change log to `Change_Type == 'PEOPLE'` (15,187 `assignee` changes, 266 `reporter` changes).
- A **true reassignment** = both `From_String` **and** `To_String` are real people.
- Spot check: an issue with `From = NaN → To = 8851` is just the *first* assignment, not a handover.
- Result: **6,805 of 20,854 issues (32.6%)** were truly reassigned → flag `Was_Reassigned`.

### 4. Measure complexity only on solo-owned issues
Keep issues that were **never reassigned** (`clean_ownership`), so the person gets credit only for work they actually owned. Per assignee:
- `Avg_Story_Point`
- `Pct_High_Priority` (share of Critical / Blocker / High, later also Major)
- `Pct_Complex_Types` (Epic / Story / Technical task)
- `Issues_Owned`

These are merged back into `core_team`.

### 5. Build groups, scores and tiers (cell 20)

**Groups**
| Group | Rule | Count |
|---|---|---|
| Core | ≥10 issues and ≥30 days tenure | 97 |
| One-off | exactly 1 issue | 85 |
| Other | everything in between | 89 |

**Scores** (min-max normalized to 0–1 across Core + One-off together):
- `Tenure_Score` (days active)
- `Volume_Score` (total issues)
- `Complexity_Score` (avg story points, missing filled with the median)
- `Priority_Score` (% high priority, missing filled with the median)
- `Seniority_Score` = **simple average of the 4**

**Tiers:** Cut points come from the **Core group only** (20/40/60/80th percentiles: ~0.248, 0.300, 0.354, 0.441), then applied to everyone. This keeps the scale anchored on people with real track records.

**Confidence flag:** `Measured` for Core, `Low (single issue)` for everyone else.

**Result:** Core splits evenly (about 20 per tier by design). One-offs land mostly in **Junior (68)**, with 15 in Mid and 2 in Junior-Mid, and none in Mid-Senior or Senior.

### 6. Sanity check on the one-offs (cell 21)
One-off `Pct_High_Priority` is only ever **0 or 100**, because it's based on one issue. So:
- Priority 0 → all 19 land in **Junior**
- Priority 100 → they land in **Junior-Mid / Mid**

This confirms their tier is driven by a single issue's priority → that's why they're marked **Low confidence**.

### 7. Why do one-off assignees exist? (cell 22)
Four checks on each one-off issue:
1. Was a core member active in the same project at that date?
2. Did the assignee report it themselves (`Self_Reported`)?
3. Was it reassigned to them?
4. Does their issue profile differ from core work?

**Profile comparison**

| Metric | One-off | Core |
|---|---|---|
| Issues | 85 | 18,557 |
| % high priority | 54.1 | 60.7 |
| % with story points | 30.6 | 46.1 |
| Avg components | 0.7 | 1.0 |
| % reassigned | 57.6 | 35.3 |

The issue **type mix is nearly identical** (about 50% Bugs in both), so it's not a different kind of work.

**Likely reason:**
| Reason | Count |
|---|---|
| Reassigned to them | 37 |
| No clear reason | 24 |
| Self-reported / own fix | 22 |
| Core team inactive then | 2 |

**Takeaway:** One-offs are mostly handoffs and self-fixes (occasional contributors), not a hidden group of specialists. So their tiers should be treated cautiously.

---

## Things worth double-checking
- **Priority definition changed:** cell 13 uses `Critical/Blocker/High`, cell 20 adds `Major`. Pick one so results are consistent.
- **Core cut points** are quantiles, so Core will always split ~evenly across tiers by construction. That's a modelling choice, not a finding.
- **One-off tiers** rest on 1 issue each; consider excluding them from rate or cost conclusions, or keeping the `Low` confidence flag visible downstream.
- **"No clear reason"** (24 one-offs) is the biggest unexplained group.

In [ ]:
user_stats = main.groupby('Assignee_ID').agg(
    Total_Issues=('ID', 'count'),
    First_Activity=('Creation_Date', 'min'),
    Last_Activity=('Creation_Date', 'max'),
    Avg_Story_Point=('Story_Point', 'mean'),
    Avg_Resolution_Minutes=('Resolution_Time_Minutes', 'mean'),
).reset_index()

user_stats['Tenure_Days'] = (user_stats['Last_Activity'] - user_stats['First_Activity']).dt.days

user_stats.describe()

,Assignee_ID,Total_Issues,First_Activity,Last_Activity,Avg_Story_Point,Avg_Resolution_Minutes,Tenure_Days
count,271.000000,271.000000,271,271,135.000000,2.710000e+02,271.000000
mean,128915.051661,70.062731,2012-07-28 00:26:37.992620,2014-06-13 13:14:14.660516,4.229593,2.588117e+05,685.191882
min,8845.000000,1.000000,2004-03-23 02:49:11,2004-08-03 06:23:13,0.000000,0.000000e+00,0.000000
25%,163665.500000,1.000000,2011-02-08 16:37:48,2011-09-11 02:33:45,2.000000,2.336608e+04,0.000000
50%,163962.000000,4.000000,2012-03-19 22:35:07,2014-09-16 14:37:12,4.000000,1.142205e+05,189.000000
75%,164947.000000,27.500000,2015-10-29 00:51:31,2017-10-01 12:33:35.500000,5.270719,2.731567e+05,991.500000
max,165814.000000,1402.000000,2020-06-09 21:41:05,2020-10-21 15:05:09,21.000000,3.548011e+06,4709.000000
std,65445.165852,193.714547,NaN,NaN,3.391519,4.435063e+05,1003.537787


In [ ]:
# Keep only assignees with meaningful, sustained involvement
core_team = user_stats[(user_stats['Total_Issues'] >= 10) & (user_stats['Tenure_Days'] >= 30)]
print(f"Core team size: {len(core_team)} out of {len(user_stats)} total assignees")
core_team.describe()

Core team size: 97 out of 271 total assignees


,Assignee_ID,Total_Issues,First_Activity,Last_Activity,Avg_Story_Point,Avg_Resolution_Minutes,Tenure_Days
count,97.000000,97.000000,97,97,71.000000,9.700000e+01,97.000000
mean,141940.958763,191.309278,2010-09-11 11:32:15.773196,2014-11-29 00:04:16.762886,4.057223,2.226394e+05,1539.020619
min,8851.000000,10.000000,2004-03-23 02:49:11,2005-07-22 18:20:15,0.000000,4.566583e+03,36.000000
25%,163675.000000,25.000000,2007-07-07 00:51:35,2011-12-28 16:45:33,3.270221,7.818737e+04,575.000000
50%,163975.000000,64.000000,2011-03-01 11:57:55,2015-02-17 23:16:06,4.200000,1.517997e+05,1203.000000
75%,164714.000000,203.000000,2013-03-04 16:27:10,2018-03-29 05:41:42,5.215000,2.590674e+05,2282.000000
max,165801.000000,1402.000000,2017-05-22 17:37:51,2020-10-21 15:05:09,8.522936,2.444538e+06,4709.000000
std,54905.168156,287.050277,NaN,NaN,1.634111,2.965810e+05,1168.732629


In [ ]:
change_log = pd.read_csv(
    'tawos_data/Change_Log.csv',
    usecols=['Issue_ID', 'Change_Type', 'Field', 'From_String', 'To_String']
)

assignee_changes = change_log[change_log['Change_Type'] == 'PEOPLE']

# True reassignment = both From and To are real people (not empty)
true_reassignments = assignee_changes[
    (assignee_changes['Field'] == 'assignee') &
    (assignee_changes['From_String'].notna()) &
    (assignee_changes['To_String'].notna())
]

main['Was_Reassigned'] = main['ID'].isin(set(true_reassignments['Issue_ID']))
print(f"Truly reassigned issues: {main['Was_Reassigned'].sum()} out of {len(main)} "
      f"({main['Was_Reassigned'].mean()*100:.1f}%)")

Truly reassigned issues: 6805 out of 20854 (32.6%)


In [ ]:
clean_ownership = main[~main['Was_Reassigned']]

complexity_stats_clean = clean_ownership.groupby('Assignee_ID').agg(
    Avg_Story_Point=('Story_Point', 'mean'),
    Pct_High_Priority=('Priority', lambda x: (x.isin(['Critical', 'Blocker', 'High'])).mean() * 100),
).reset_index()
print(assignee_changes['Field'].value_counts())

Field
assignee    15187
reporter      266
Name: count, dtype: int64


In [ ]:
tables = ['Change_Log', 'Comment', 'Issue_Component', 'Issue_Link', 'Sprint', 'Project']
dataframes = {t: pd.read_csv(f'tawos_data/{t}.csv') for t in tables}
# Look at a few reassigned issues' actual change history to understand the pattern
sample_issue = assignee_changes['Issue_ID'].iloc[0]
dataframes['Change_Log'][
    (dataframes['Change_Log']['Issue_ID'] == sample_issue) &
    (dataframes['Change_Log']['Change_Type'] == 'PEOPLE')
][['Field', 'From_String', 'To_String', 'Creation_Date']]

,Field,From_String,To_String,Creation_Date
18,assignee,NaN,8851,2020-08-11 10:06:03


In [ ]:
# True reassignment = both From and To are actual people (not NaN/empty)
true_reassignments = assignee_changes[
    (assignee_changes['Field'] == 'assignee') &
    (assignee_changes['From_String'].notna()) &
    (assignee_changes['To_String'].notna())
]

true_reassigned_ids = set(true_reassignments['Issue_ID'].unique())
main['Was_Reassigned'] = main['ID'].isin(true_reassigned_ids)

print(f"Truly reassigned issues: {main['Was_Reassigned'].sum()} out of {len(main)} ({main['Was_Reassigned'].mean()*100:.1f}%)")

Truly reassigned issues: 6805 out of 20854 (32.6%)


In [ ]:
clean_ownership = main[~main['Was_Reassigned']]

complexity_stats = clean_ownership.groupby('Assignee_ID').agg(
    Avg_Story_Point=('Story_Point', 'mean'),
    Pct_High_Priority=('Priority', lambda x: (x.isin(['Critical', 'Blocker', 'High'])).mean() * 100),
    Pct_Complex_Types=('Type', lambda x: (x.isin(['Epic', 'Story', 'Technical task'])).mean() * 100),
    Issues_Owned=('ID', 'count'),
).reset_index()

core_team = core_team.drop(columns=[c for c in ['Avg_Story_Point', 'Pct_High_Priority', 'Pct_Bugs', 'Pct_Complex_Types'] if c in core_team.columns])
core_team = core_team.merge(complexity_stats, on='Assignee_ID', how='left')

print(core_team.columns.tolist())  # Confirm Pct_High_Priority is now there

['Assignee_ID', 'Total_Issues', 'First_Activity', 'Last_Activity', 'Avg_Resolution_Minutes', 'Tenure_Days', 'Avg_Story_Point', 'Pct_High_Priority', 'Pct_Complex_Types', 'Issues_Owned']


In [ ]:
# ---------- 1. Groups ----------
user_stats['Group'] = np.select(
    [(user_stats['Total_Issues'] >= 10) & (user_stats['Tenure_Days'] >= 30),
     user_stats['Total_Issues'] == 1],
    ['Core', 'One-off'], default='Other')
print(user_stats['Group'].value_counts())

pool = user_stats[user_stats['Group'].isin(['Core', 'One-off'])].copy()

# ---------- 2. Solo-owned metrics (issues never reassigned) ----------
clean_ownership = main[~main['Was_Reassigned']]
cx = clean_ownership.groupby('Assignee_ID').agg(
    Avg_Story_Point=('Story_Point', 'mean'),
    Pct_High_Priority=('Priority', lambda x: x.isin(['Critical', 'Blocker', 'High', 'Major']).mean() * 100),
).reset_index()
pool = pool.drop(columns=['Avg_Story_Point']).merge(cx, on='Assignee_ID', how='left')

# ---------- 3. Scores (same 4 as core_team, normalized on Core + One-off together) ----------
def normalize(col): return (col - col.min()) / (col.max() - col.min())

pool['Tenure_Score']     = normalize(pool['Tenure_Days'])
pool['Volume_Score']     = normalize(pool['Total_Issues'])
pool['Complexity_Score'] = normalize(pool['Avg_Story_Point'].fillna(pool['Avg_Story_Point'].median()))
pool['Priority_Score']   = normalize(pool['Pct_High_Priority'].fillna(pool['Pct_High_Priority'].median()))
pool['Seniority_Score']  = pool[['Tenure_Score', 'Volume_Score',
                                 'Complexity_Score', 'Priority_Score']].mean(axis=1)

# ---------- 4. Tiers: cut points from Core only, applied to everyone ----------
TIER_LABELS = ['Junior', 'Junior-Mid', 'Mid', 'Mid-Senior', 'Senior']
edges = pool.loc[pool['Group'] == 'Core', 'Seniority_Score'].quantile([0.2, 0.4, 0.6, 0.8]).tolist()
pool['Seniority_Tier'] = pd.cut(pool['Seniority_Score'],
                                bins=[-np.inf] + edges + [np.inf],
                                labels=TIER_LABELS)

# ---------- 5. Flag how much evidence each tier rests on ----------
pool['Tier_Confidence'] = np.where(pool['Group'] == 'Core', 'Measured', 'Low (single issue)')

print(edges)
print(pd.crosstab(pool['Group'], pool['Seniority_Tier']))

Group
Core       97
Other      89
One-off    85
Name: count, dtype: int64
[0.24836345712748348, 0.2997816573709634, 0.3542177878648989, 0.4413649680748749]
Seniority_Tier  Junior  Junior-Mid  Mid  Mid-Senior  Senior
Group                                                      
Core                20          19   19          19      20
One-off             68           2   15           0       0


In [ ]:
oo = pool[pool['Group'] == 'One-off']
print(pd.crosstab(oo['Seniority_Tier'], oo['Pct_High_Priority'].round(0)))

Pct_High_Priority  0.0    100.0
Seniority_Tier                 
Junior                19      0
Junior-Mid             0      2
Mid                    0     15


In [ ]:
one_ids  = pool.loc[pool['Group'] == 'One-off', 'Assignee_ID']
core_ids = pool.loc[pool['Group'] == 'Core', 'Assignee_ID']
one          = main[main['Assignee_ID'].isin(one_ids)].copy()
core_issues  = main[main['Assignee_ID'].isin(core_ids)]

# 1. Were core members active in the same project when the issue was created?
win = (core_issues.groupby(['Project_ID', 'Assignee_ID'])['Creation_Date']
       .agg(['min', 'max']).reset_index())

def n_core_active(r):
    w = win[win['Project_ID'] == r['Project_ID']]
    return ((w['min'] <= r['Creation_Date']) & (w['max'] >= r['Creation_Date'])).sum()

one['Core_Active_Same_Project'] = one.apply(n_core_active, axis=1)

# 2. Did the assignee report it themselves?  3. Was it reassigned?
one['Self_Reported'] = one['Reporter_ID'] == one['Assignee_ID']

# 4. Different kind of work? Compare issue profiles
def profile(df):
    return pd.Series({
        'Issues': len(df),
        'Pct_High_Priority': df['Priority'].isin(['Critical', 'Blocker', 'High', 'Major']).mean() * 100,
        'Pct_Planned (has story pt)': df['Story_Point'].notna().mean() * 100,
        'Avg_Components': df['Component_Count'].mean(),
        'Pct_Reassigned': df['Was_Reassigned'].mean() * 100,
    })
print(pd.DataFrame({'One-off': profile(one), 'Core': profile(core_issues)}).round(1))
print(pd.concat([one['Type'].value_counts(normalize=True),
                 core_issues['Type'].value_counts(normalize=True)],
                axis=1, keys=['One-off', 'Core']).round(3))

# Verdict per issue, first matching reason wins
one['Likely_Reason'] = np.select(
    [one['Core_Active_Same_Project'] == 0, one['Self_Reported'], one['Was_Reassigned']],
    ['Core team inactive then', 'Self-reported / own fix', 'Reassigned to them'],
    default='No clear reason')
print(one['Likely_Reason'].value_counts())

                            One-off     Core
Issues                         85.0  18557.0
Pct_High_Priority              54.1     60.7
Pct_Planned (has story pt)     30.6     46.1
Avg_Components                  0.7      1.0
Pct_Reassigned                 57.6     35.3
                     One-off   Core
Type                               
Bug                    0.506  0.495
Enhancement Request    0.141  0.131
Task                   0.118  0.132
Story                  0.106  0.076
Improvement            0.082  0.088
New Feature            0.024  0.037
Sub-task               0.024  0.019
Technical task           NaN  0.015
Epic                     NaN  0.006
Likely_Reason
Reassigned to them         37
No clear reason            24
Self-reported / own fix    22
Core team inactive then     2
Name: count, dtype: int64


## HRA dataset Exploration

In [ ]:
hra = pd.read_csv('enterprise_hra_allocation_dataset.csv')
hra.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Employee_ID                   5000 non-null   str    
 1   Age                           5000 non-null   int64  
 2   Gender                        5000 non-null   str    
 3   Location                      5000 non-null   str    
 4   Department                    5000 non-null   str    
 5   Job_Role                      5000 non-null   str    
 6   Education_Level               5000 non-null   str    
 7   Employment_Type               5000 non-null   str    
 8   Shift_Type                    5000 non-null   str    
 9   Experience_Years              5000 non-null   float64
 10  Working_Hours_Monthly         5000 non-null   float64
 11  Overtime_Hours_Monthly        5000 non-null   float64
 12  Attendance_Rate               5000 non-null   float64
 13  Leave_Days_Qua

In [ ]:
hra['Job_Role'].value_counts()

Job_Role
Operations Executive          243
Business Development          225
Sales Executive               223
Account Manager               221
Customer Success Executive    214
Process Analyst               206
System Administrator          199
Shift Supervisor              195
QA Engineer                   184
Support Lead                  184
Data Analyst                  181
AI Engineer                   174
Support Associate             164
Software Engineer             164
Inventory Analyst             137
Procurement Executive         132
Logistics Coordinator         131
Payroll Specialist            130
Innovation Specialist         127
SEO Analyst                   122
Research Associate            119
Brand Associate               118
Product Analyst               115
Accountant                    108
Content Strategist            107
Recruiter                     106
Marketing Executive           106
Audit Associate               103
Financial Analyst              99
Train

In [ ]:
tech_roles = ['Software Engineer', 'QA Engineer', 'AI Engineer', 'Data Analyst', 'System Administrator']

hra_tech = hra[hra['Job_Role'].isin(tech_roles)]
print(f"Tech roles pool size: {len(hra_tech)}")
print(hra_tech['Job_Role'].value_counts())
print()
print(hra_tech['Experience_Years'].describe())

Tech roles pool size: 902
Job_Role
System Administrator    199
QA Engineer             184
Data Analyst            181
AI Engineer             174
Software Engineer       164
Name: count, dtype: int64

count    902.000000
mean       4.656707
std        3.104413
min        0.170000
25%        2.310000
50%        3.975000
75%        6.387500
max       18.000000
Name: Experience_Years, dtype: float64


In [ ]:
print(hra_tech['Education_Level'].value_counts())

Education_Level
Bachelor    435
Master      241
MBA         124
Diploma      71
PhD          31
Name: count, dtype: int64


In [ ]:
hra_tech = hra_tech.copy()

def normalize(col):
    return (col - col.min()) / (col.max() - col.min())

hra_tech['Exp_Norm'] = normalize(hra_tech['Experience_Years'])
hra_tech['Productivity_Norm'] = normalize(hra_tech['Productivity_Score'])
hra_tech['Training_Norm'] = normalize(hra_tech['Training_Score'])
hra_tech['Adaptability_Norm'] = normalize(hra_tech['Adaptability_Score'])
hra_tech['Collaboration_Norm'] = normalize(hra_tech['Collaboration_Score'])
hra_tech['Efficiency_Norm'] = normalize(hra_tech['Task_Completion_Efficiency'])

weights = {
    'Productivity_Norm': 3.0,
    'Exp_Norm': 2.5,
    'Efficiency_Norm': 2.0,
    'Training_Norm': 1.0,
    'Adaptability_Norm': 1.0,
    'Collaboration_Norm': 1.0,
}
total_weight = sum(weights.values())

hra_tech['HRA_Seniority_Score'] = sum(
    hra_tech[col] * w for col, w in weights.items()
) / total_weight

hra_tech['HRA_Seniority_Tier'] = pd.qcut(hra_tech['HRA_Seniority_Score'], q=3, labels=['Junior', 'Mid', 'Senior'])
print(hra_tech['HRA_Seniority_Tier'].value_counts())

HRA_Seniority_Tier
Junior    301
Senior    301
Mid       300
Name: count, dtype: int64


In [ ]:
core_dev_roles = ['Software Engineer', 'QA Engineer', 'System Administrator']
hra_final_pool = hra_tech[hra_tech['Job_Role'].isin(core_dev_roles)]
print(hra_final_pool['Job_Role'].value_counts())
print()
print(hra_final_pool['HRA_Seniority_Tier'].value_counts())

Job_Role
System Administrator    199
QA Engineer             184
Software Engineer       164
Name: count, dtype: int64

HRA_Seniority_Tier
Senior    192
Mid       184
Junior    171
Name: count, dtype: int64


In [ ]:
groups = {
    'Performance': {'Productivity_Score': 3.0, 'KPI_Score': 2.5, 'Performance_Rating': 2.0,
                    'Employee_Efficiency_Score': 2.0, 'Task_Completion_Efficiency': 2.0,
                    'Task_Completion_Rate': 1.0},
    'Experience':  {'Experience_Years': 2.5, 'Skill_Score': 2.0},
    'Growth':      {'Training_Score': 1.0, 'Training_Effectiveness_Score': 1.0, 'Adaptability_Score': 1.0},
    'Reliability': {'Collaboration_Score': 1.0, 'Attendance_Rate': 1.0, 'Absenteeism_Risk_Score': 1.0},
}
INVERT = {'Absenteeism_Risk_Score'}   # higher = worse; confirm with the describe() below

all_cols = [c for g in groups.values() for c in g]
print(hra_tech[all_cols].describe().T[['min', 'max', 'mean']])
print(hra_tech[all_cols].corr().round(2))   # look for pairs above ~0.8

                                  min       max       mean
Productivity_Score            63.2700  100.0000  87.900820
KPI_Score                     52.3900  100.0000  81.631120
Performance_Rating             2.6200    5.0000   4.081596
Employee_Efficiency_Score     64.3000  100.0000  85.577517
Task_Completion_Efficiency     0.0104    0.1245   0.049208
Task_Completion_Rate          45.4400  100.0000  87.200144
Experience_Years               0.1700   18.0000   4.656707
Skill_Score                   61.9000   96.7800  78.966075
Training_Score                53.9300  100.0000  78.722095
Training_Effectiveness_Score   0.0000   31.7279   1.258936
Adaptability_Score            51.6900  100.0000  79.608160
Collaboration_Score           50.6500  100.0000  76.831796
Attendance_Rate               65.6200  100.0000  88.060576
Absenteeism_Risk_Score         0.0000   58.5700  18.734324
                              Productivity_Score  KPI_Score  \
Productivity_Score                          1.00    

## HRA seniority score: choosing the inputs

The HRA dataset has 40 columns. I used only the ones that describe a person's **track record**
(performance, experience, growth, reliability). I left out the columns that describe their
**current situation** (workload, availability, utilization, task capacity, allocation suitability).
Someone who is busy this month is not less senior, so those columns are saved for a later
"allocation readiness" step.

Before building the score I checked how the candidate columns relate to each other
(correlation matrix). Two columns that move together carry the same information, and counting
both would give that information double weight.

| Finding | Decision | Why |
|---|---|---|
| `KPI_Score` and `Performance_Rating` correlate at **1.00**. The rating runs 2.6-5.0 and the KPI 52-100, so the rating is the KPI divided by 20 | Dropped `Performance_Rating` | They are the same number on a different scale |
| `Attendance_Rate` and `Absenteeism_Risk_Score` correlate at **-0.81** | Kept `Attendance_Rate`, dropped `Absenteeism_Risk_Score` | They overlap heavily, and this avoids having to flip an inverted score |
| `Employee_Efficiency_Score` correlates **0.82** with `Productivity_Score` and **0.81** with `Task_Completion_Rate` | Dropped `Employee_Efficiency_Score` | It appears to be built from those two (inferred from the correlations, not confirmed), so keeping all three would count them twice |
| `Training_Effectiveness_Score` has a mean of 1.26 but a maximum of 31.7 | Kept it, but scaled by **percentile rank** instead of min-max | It is very skewed. Min-max scaling would push almost everyone close to 0 and hide the differences between them |
| `Skill_Score` correlates 0.74 with `KPI_Score` and 0.63 with `Training_Score` | Kept it | The relationship is moderate, and the columns sit in different groups (Experience vs. Performance and Growth) |

### The four groups and their weights
The 11 remaining columns are combined into four group scores. Inside each group the weights are
a weighted average, and the min-max scaling puts every column on a 0-1 scale first.

| Group | Columns |
|---|---|
| Performance | Productivity, KPI, Task completion efficiency, Task completion rate |
| Experience | Experience years, Skill score |
| Growth | Training score, Training effectiveness, Adaptability |
| Reliability | Collaboration, Attendance rate |

The weights are a judgment call. They start from the weights of the earlier version of this
analysis (productivity highest, then experience and efficiency, the rest equal), and there is
no ground truth to calibrate them against.

In [ ]:
def norm(s): return (s - s.min()) / (s.max() - s.min())

hra_tech = hra_tech.copy()
for g, cols in groups.items():
    total = sum(cols.values())
    hra_tech[f'{g}_Score'] = sum(
        (1 - norm(hra_tech[c]) if c in INVERT else norm(hra_tech[c])) * w
        for c, w in cols.items()) / total

In [ ]:
groups = {
    'Performance': {'Productivity_Score': 3.0, 'KPI_Score': 2.5,
                    'Task_Completion_Efficiency': 2.0, 'Task_Completion_Rate': 1.0},
    'Experience':  {'Experience_Years': 2.5, 'Skill_Score': 2.0},
    'Growth':      {'Training_Score': 1.0, 'Training_Effectiveness_Score': 1.0,
                    'Adaptability_Score': 1.0},
    'Reliability': {'Collaboration_Score': 1.0, 'Attendance_Rate': 1.0},
}
RANKED = {'Training_Effectiveness_Score'}   # skewed: use percentile rank instead of min-max

def norm(s): return (s - s.min()) / (s.max() - s.min())

hra_tech = hra_tech.copy()
for g, cols in groups.items():
    total = sum(cols.values())
    parts = [(hra_tech[c].rank(pct=True) if c in RANKED else norm(hra_tech[c])) * w
             for c, w in cols.items()]
    hra_tech[f'{g}_Score'] = sum(parts) / total

group_cols = [f'{g}_Score' for g in groups]
print(hra_tech[group_cols].describe().round(2))
print(hra_tech[group_cols].corr().round(2))

       Performance_Score  Experience_Score  Growth_Score  Reliability_Score
count             902.00            902.00        902.00             902.00
mean                0.59              0.36          0.54               0.59
std                 0.12              0.14          0.13               0.14
min                 0.21              0.05          0.20               0.15
25%                 0.50              0.25          0.45               0.50
50%                 0.58              0.34          0.54               0.59
75%                 0.67              0.44          0.63               0.69
max                 0.88              0.84          0.86               0.96
                   Performance_Score  Experience_Score  Growth_Score  \
Performance_Score               1.00              0.32          0.37   
Experience_Score                0.32              1.00          0.51   
Growth_Score                    0.37              0.51          1.00   
Reliability_Score           

## Group scores: what the distributions tell us

The four group scores are not spread evenly over 0-1. Averaging several columns pulls scores
toward the middle, and Experience in particular is low overall (mean 0.36, maximum 0.84).

If "low / mid / high" were defined by splitting 0-1 into
equal thirds, almost nobody would count as "high Experience", and the fuzzy rules would
rarely fire. The system would behave like a plain average. Instead, each group's sets are
anchored on **its own distribution**:
- **low:** fully low up to the 20th percentile
- **mid:** peaks at the median
- **high:** fully high from the 80th percentile

So "high Experience" means high *compared with this pool of employees*.

**Independence of the groups.** Performance, Experience and Growth correlate only 0.3-0.5 with
each other, and Reliability is close to 0 with all three. Each group therefore adds information
the others don't have, which is what the combination rules need. If the groups had been highly
correlated, most of the 81 rules (3 levels across 4 groups) would never be triggered.

In [ ]:
import itertools, time
import simpful as sf

TERMS = ['low', 'mid', 'high']
OUT   = ['junior', 'junior_mid', 'mid', 'mid_senior', 'senior']
W     = {'Performance': 0.35, 'Experience': 0.30, 'Growth': 0.20, 'Reliability': 0.15}

FS = sf.FuzzySystem(show_banner=False)
for g in groups:
    p20, p50, p80 = hra_tech[f'{g}_Score'].quantile([0.2, 0.5, 0.8])
    FS.add_linguistic_variable(g, sf.LinguisticVariable([
        sf.FuzzySet(function=sf.Trapezoidal_MF(-1, 0, p20, p50), term='low'),   # full "low" up to p20
        sf.FuzzySet(function=sf.Triangular_MF(p20, p50, p80),   term='mid'),    # peaks at the median
        sf.FuzzySet(function=sf.Trapezoidal_MF(p50, p80, 1, 2), term='high'),   # full "high" from p80
    ], universe_of_discourse=[0, 1]))
FS.add_linguistic_variable('Seniority', sf.AutoTriangle(5, terms=OUT, universe_of_discourse=[0, 1]))

rules = []
for combo in itertools.product(range(3), repeat=4):             # 81 rules
    level = sum(W[g] * l for g, l in zip(groups, combo)) / 2     # 0..1
    idx = min(int(level * 5), 4)
    if combo[1] == 0:                                            # gate: low Experience caps at 'mid'
        idx = min(idx, 2)
    cond = ' AND '.join(f'({g} IS {TERMS[l]})' for g, l in zip(groups, combo))
    rules.append(f'IF {cond} THEN (Seniority IS {OUT[idx]})')
FS.add_rules(rules)

def fuzzy_seniority(row):
    try:
        for g in groups:
            FS.set_variable(g, float(row[f'{g}_Score']))
        return FS.Mamdani_inference(['Seniority'])['Seniority']
    except Exception:
        return np.nan

# --- test on 50 rows first ---
t = time.time()
test = hra_tech.head(50).apply(fuzzy_seniority, axis=1)
print(test.describe().round(3))
print(f"{time.time() - t:.1f}s for 50 rows, {test.isna().sum()} failures")

count    50.000
mean      0.470
std       0.223
min       0.088
25%       0.305
50%       0.449
75%       0.672
max       0.901
dtype: float64
4.0s for 50 rows, 0 failures


## HRA seniority with fuzzy logic

### Why fuzzy logic
A weighted average has hard edges: a score of 0.61 is Senior and 0.60 is Mid, and the model can't say
"this person is a bit of both". Fuzzy logic lets a score belong partly to two levels, and it lets me
write **rules** that combine measures in ways an average can't, such as "high performance but little
experience is still not senior".

### How the system is built
1. **Four group scores** (Performance, Experience, Growth, Reliability) come from the 11 HRA columns, as described above.
2. **Fuzzy inputs.** Each group score is described as *low*, *mid* or *high*. These sets are anchored on the
   data's own percentiles (full "low" up to the 20th, peak "mid" at the median, full "high" from the 80th),
   because the scores are bunched in the middle of 0-1.
3. **81 rules**, one for every combination of low/mid/high across the four groups (3⁴). Each rule's
   result is a seniority level between *junior* and *senior*, calculated from the combination weighted
   by Performance 0.35, Experience 0.30, Growth 0.20 and Reliability 0.15.
4. **One gate:** anyone with low Experience is capped at *mid*, however high the other scores are.
5. **Inference and output.** Mamdani inference combines all the rules that apply and returns a single
   seniority score between 0 and 1.
6. **Tiers.** That score is split into five equal-sized groups (quintiles) with the same labels used for
   TAWOS: Junior, Junior-Mid, Mid, Mid-Senior, Senior.

Library: **Simpful** (rules are written in near-natural language, which keeps the logic readable).
The weights and the gate are judgment calls, since there is no ground truth to calibrate them against.
Technical note: the Simpful start-up banner needs a module that newer Python versions no longer include,
so the system is created with `show_banner=False`.

In [ ]:
hra_tech['HRA_Seniority_Score'] = hra_tech.apply(fuzzy_seniority, axis=1)
print(hra_tech['HRA_Seniority_Score'].isna().sum(), "failures")

TIER_LABELS = ['Junior', 'Junior-Mid', 'Mid', 'Mid-Senior', 'Senior']
hra_tech['HRA_Seniority_Tier'] = pd.qcut(hra_tech['HRA_Seniority_Score'], q=5, labels=TIER_LABELS)
print(hra_tech['HRA_Seniority_Tier'].value_counts())

0 failures
HRA_Seniority_Tier
Junior        181
Senior        181
Junior-Mid    180
Mid           180
Mid-Senior    180
Name: count, dtype: int64


In [ ]:
hra_tech['Linear_Score'] = sum(W[g] * hra_tech[f'{g}_Score'] for g in groups)
hra_tech['Linear_Tier'] = pd.qcut(hra_tech['Linear_Score'], q=5, labels=TIER_LABELS)
print("Same tier as linear:", (hra_tech['Linear_Tier'] == hra_tech['HRA_Seniority_Tier']).mean().round(2))
print(pd.crosstab(hra_tech['Linear_Tier'], hra_tech['HRA_Seniority_Tier']))

Same tier as linear: 0.75
HRA_Seniority_Tier  Junior  Junior-Mid  Mid  Mid-Senior  Senior
Linear_Tier                                                    
Junior                 160          20    1           0       0
Junior-Mid              21         134   25           0       0
Mid                      0          26  123          31       0
Mid-Senior               0           0   27         118      35
Senior                   0           0    4          31     146


## Fuzzy result vs. a plain weighted average

All 902 employees were scored with no failures. The five tiers hold about 180 people each. That is
by construction (quintiles).

**Comparison.** To see what the fuzzy rules add, a calculation has been done of a plain weighted average of the same four
group scores, with the same weights, and split it into the same five tiers.

- About three quarters of employees land in the same tier under both methods.
- **221 of 902 (about a quarter) change tier**: 216 move one step and 5 move two steps.
- Moves go in both directions, so the rules do more than lower people.

**How to read this.** The fuzzy score mostly agrees with the average and mainly reshuffles people
close to a tier boundary, where the rule combinations matter.

The five largest changes (two steps) show that the Experience gate is only part of the story.
Of the four Senior → Mid cases, one has low Experience (0.23 vs. a pool median of 0.34), which is
consistent with the gate. The other three have above-median Experience, but two have below-median
Performance (0.47 vs. 0.58) and the third has low Reliability (0.37 vs. 0.59). Performance carries
the most weight in the rules, and the fuzzy sets are anchored on percentiles, so a weak group score
pulls a person down more than it does in a plain average, where strong scores in other groups can
offset it.
Three of these four were borderline Seniors in the plain average (within 0.01 of the cutoff
of 0.583). The fourth (0.628) was further inside Senior and dropped because its Performance
score is well below the median.

In [ ]:
core_dev_roles = ['Software Engineer', 'QA Engineer', 'System Administrator']
hra_final_pool = hra_tech[hra_tech['Job_Role'].isin(core_dev_roles)]

check = pd.DataFrame({
    'TAWOS_needed':  pool['Seniority_Tier'].value_counts(),
    'HRA_available': hra_final_pool['HRA_Seniority_Tier'].value_counts(),
}).reindex(TIER_LABELS)
check['Enough'] = check['HRA_available'] >= check['TAWOS_needed']
print(check)

            TAWOS_needed  HRA_available  Enough
Junior                88            104    True
Junior-Mid            21            110    True
Mid                   34            108    True
Mid-Senior            19            120    True
Senior                20            105    True


In [ ]:
np.random.seed(42)
used, rows = set(), []

for _, row in pool.iterrows():
    cands = hra_final_pool[(hra_final_pool['HRA_Seniority_Tier'] == row['Seniority_Tier']) &
                           (~hra_final_pool['Employee_ID'].isin(used))]
    if cands.empty:
        raise ValueError(f"No HRA employees left for tier {row['Seniority_Tier']}")
    m = cands.sample(1).iloc[0]
    used.add(m['Employee_ID'])
    rows.append({'Assignee_ID': row['Assignee_ID'], 'HRA_Employee_ID': m['Employee_ID'],
                 'Job_Role': m['Job_Role'], 'Department': m['Department'],
                 'Experience_Years': m['Experience_Years'],
                 'HRA_Seniority_Tier': m['HRA_Seniority_Tier']})

team = pool.merge(pd.DataFrame(rows), on='Assignee_ID', how='left')

print("All tiers match:", (team['Seniority_Tier'].astype(str) == team['HRA_Seniority_Tier'].astype(str)).all())
print(team['HRA_Employee_ID'].nunique(), "unique HRA employees for", len(team), "people")
print(pd.crosstab(team['Group'], team['Job_Role']))

All tiers match: True
182 unique HRA employees for 182 people
Job_Role  QA Engineer  Software Engineer  System Administrator
Group                                                         
Core               44                 24                    29
One-off            28                 24                    33


## Matching TAWOS contributors to HRA employees

**Goal:** give every TAWOS contributor in the pool (core team + one-offs, 182 people) a stand-in
employee from the HRA dataset with the **same seniority tier**, so that rates and budgets can be
simulated later.

### Procedure
1. **Candidate pool:** HRA employees in three roles (Software Engineer, QA Engineer, System
   Administrator), the closest to the work in the Jira projects.
2. **Supply check first.** Matching is one-to-one, so each tier needs at least as many HRA employees
   as TAWOS contributors.
3. **Match:** for each contributor, draw one unused HRA employee from the same tier at random
   (fixed seed 42, so the result is reproducible). Nobody is used twice.
4. **Verify:** every match is in the same tier, and the number of unique HRA employees equals the
   number of contributors.

Tier is the only attribute used to match: TAWOS has no job role, department or experience
information to match on.

### Saved files
- `team_tier_matching.csv`: each TAWOS contributor with their HRA stand-in.
- `hra_tech_scored.csv`: the HRA pool with group scores, fuzzy score and tier. Employees who were not
  picked are labelled *Not Selected*.

### Limits to keep in mind
- **Tiers are relative, and the two sides are different populations.** TAWOS tiers use the core team's
  cut points. HRA tiers are quintiles of the HRA tech pool. "Mid" on each side means "middle of its own
  group", not the same absolute skill.
- **The pairing is arbitrary inside a tier.** Which HRA employee stands for which contributor is
  random, so conclusions should be drawn at tier or role level, not about individual people.
- **One-off contributors carry low-confidence tiers** (see above), and that carries over to their match.
- **The 89 contributors in the "Other" group get no tier and no HRA match.** Their issues have no
  hourly rate yet. This is handled in the rates step.

In [ ]:
hra_tech['TAWOS_Assignee_ID'] = (hra_tech['Employee_ID']
    .map(dict(zip(team['HRA_Employee_ID'], team['Assignee_ID']))).fillna('Not Selected'))

team.to_csv('tawos_data/team_tier_matching.csv', index=False)
hra_tech.to_csv('tawos_data/hra_tech_scored.csv', index=False)

In [ ]:
movers = hra_tech[(hra_tech['Linear_Tier'] == 'Senior') & (hra_tech['HRA_Seniority_Tier'] == 'Mid')]
print(movers[['Experience_Score', 'Performance_Score', 'Growth_Score', 'Reliability_Score']].round(2))
print("Median Experience_Score, whole pool:", hra_tech['Experience_Score'].median().round(2))

      Experience_Score  Performance_Score  Growth_Score  Reliability_Score
1872              0.76               0.47          0.78               0.52
2862              0.23               0.75          0.73               0.76
3257              0.68               0.47          0.63               0.60
4780              0.69               0.61          0.57               0.37
Median Experience_Score, whole pool: 0.34


In [ ]:
cut = hra_tech['Linear_Score'].quantile(0.8)
print("Senior cutoff (linear):", cut.round(3))
print(hra_tech.loc[movers.index, 'Linear_Score'].round(3))

Senior cutoff (linear): 0.583
1872    0.628
2862    0.593
3257    0.584
4780    0.589
Name: Linear_Score, dtype: float64


In [ ]:
rate_bands = {'Junior': (15, 20), 'Junior-Mid': (20, 26), 'Mid': (26, 33),
              'Mid-Senior': (33, 42), 'Senior': (42, 55)}

def assign_rate(r, grp):
    low, high = rate_bands[r['Seniority_Tier']]
    t = grp.loc[grp['Seniority_Tier'] == r['Seniority_Tier'], 'Seniority_Score']
    span = t.max() - t.min()
    pos = 0.5 if span == 0 else (r['Seniority_Score'] - t.min()) / span   # guards identical scores
    return round(low + pos * (high - low), 2)

team['Hourly_Rate_USD'] = team.apply(lambda r: assign_rate(r, team), axis=1)
print(team.groupby('Seniority_Tier', observed=True)['Hourly_Rate_USD'].describe().round(2))

                count   mean   std   min    25%    50%    75%   max
Seniority_Tier                                                     
Junior           88.0  18.81  1.53  15.0  17.60  19.79  19.79  20.0
Junior-Mid       21.0  23.50  1.74  20.0  22.40  23.93  24.72  26.0
Mid              34.0  29.76  1.97  26.0  29.33  29.33  31.60  33.0
Mid-Senior       19.0  37.78  3.09  33.0  34.97  37.26  40.70  42.0
Senior           20.0  45.91  3.79  42.0  42.80  44.98  48.46  55.0


In [ ]:
drop_helpers = ['Linear_Score', 'Linear_Tier', 'TAWOS_Assignee_ID']
hra_cols = [c for c in hra_tech.columns if c not in team.columns and c not in drop_helpers]
assert 'Employee_ID' in hra_cols and len(hra_cols) == len(set(hra_cols))   # no duplicates

team_profiles = team.merge(hra_tech[hra_cols], left_on='HRA_Employee_ID',
                           right_on='Employee_ID', how='inner')
assert len(team_profiles) == len(team), "some matches were lost in the merge"

team_profiles['Assignee_ID'] = team_profiles['Assignee_ID'].astype(int)
team_profiles.to_csv('tawos_data/team_profiles.csv', index=False)
print(team_profiles.shape)

(182, 70)


In [ ]:
print("All tiers match:", (team_profiles['Seniority_Tier'].astype(str) ==
                           team_profiles['HRA_Seniority_Tier'].astype(str)).all())
print(team_profiles['HRA_Employee_ID'].nunique(), "unique HRA employees")
print(team_profiles.groupby('Group')['Hourly_Rate_USD'].describe().round(2))

All tiers match: True
182 unique HRA employees
         count   mean    std   min    25%    50%    75%    max
Group                                                         
Core      97.0  31.34  10.13  17.1  22.40  31.39  40.84  55.00
One-off   85.0  20.66   4.25  15.0  19.79  19.79  19.79  31.55


In [ ]:
rates = team[['Assignee_ID', 'Hourly_Rate_USD']].copy()
rates['Rate_Source'] = 'Tier-based'

median_rate = team.loc[team['Group'] == 'Core', 'Hourly_Rate_USD'].median()
others = user_stats.loc[user_stats['Group'] == 'Other', ['Assignee_ID']].copy()
others['Hourly_Rate_USD'] = median_rate
others['Rate_Source'] = 'Assumed (core median)'

rates = pd.concat([rates, others], ignore_index=True)
main = main.merge(rates, on='Assignee_ID', how='left')
main['Rate_Source'] = main['Rate_Source'].fillna('No assignee (no cost)')

print(main['Rate_Source'].value_counts())

Rate_Source
Tier-based               18642
No assignee (no cost)     1867
Assumed (core median)      345
Name: count, dtype: int64


In [ ]:
un = main[main['Assignee_ID'].isna()].copy()
print(len(un), "unassigned issues\n")

# 1. By project: count and share of each project's issues
by_proj = pd.DataFrame({'Unassigned': un['Project_Name'].value_counts(),
                        'Total': main['Project_Name'].value_counts()})
by_proj['Pct_of_project'] = (by_proj['Unassigned'] / by_proj['Total'] * 100).round(1)
print(by_proj, "\n")

# 2. By status: overall, then per project (row %)
print(un['Status'].value_counts(), "\n")
print((pd.crosstab(un['Project_Name'], un['Status'], normalize='index') * 100).round(1), "\n")

# 3. Does it matter for cost? Effort recorded on these issues
print("Effort on unassigned issues (hours):", round(un['Total_Effort_Minutes'].sum() / 60))
print("Share of all effort:", round(un['Total_Effort_Minutes'].sum() / main['Total_Effort_Minutes'].sum() * 100, 1), "%")
print(un.groupby('Status')['Total_Effort_Minutes'].agg(['count', 'sum']).round(0))

1867 unassigned issues

                          Unassigned  Total  Pct_of_project
Project_Name                                               
Mule                            1838  11816            15.6
Appcelerator Studio               18   5979             0.3
Titanium Mobile Platform          11   3059             0.4 

Status
Closed         1307
Done            280
Open            153
New             118
Resolved          7
In Progress       2
Name: count, dtype: int64 

Status                    Closed  Done  In Progress  New  Open  Resolved
Project_Name                                                            
Appcelerator Studio         33.3   0.0          0.0  0.0  61.1       5.6
Mule                        70.6  15.2          0.1  6.4   7.3       0.3
Titanium Mobile Platform    36.4   0.0          0.0  0.0  63.6       0.0 

Effort on unassigned issues (hours): 201851
Share of all effort: 8.1 %
             count        sum
Status                       
Closed        1307  5

In [ ]:
proj_end = main.groupby('Project_Name')['Creation_Date'].max()
op = un[un['Status'].isin(['Open', 'New'])].copy()
op['Days_Before_End'] = (op['Project_Name'].map(proj_end) - op['Creation_Date']).dt.days
print(op.groupby('Project_Name')['Days_Before_End'].describe().round(0))
print(op.groupby(op['Creation_Date'].dt.year).size())

                          count    mean    std     min     25%     50%  \
Project_Name                                                             
Appcelerator Studio        11.0  1748.0  513.0  1098.0  1455.0  1489.0   
Mule                      253.0  1388.0  273.0   238.0  1201.0  1321.0   
Titanium Mobile Platform    7.0   417.0  388.0   186.0   279.0   293.0   

                             75%     max  
Project_Name                              
Appcelerator Studio       2088.0  2652.0  
Mule                      1539.0  1982.0  
Titanium Mobile Platform   293.0  1293.0  
Creation_Date
2013      1
2014      2
2015     53
2016     93
2017    114
2018      1
2019      5
2020      2
dtype: int64


In [ ]:
two = res[res['Total_Effort_Minutes'] > 0]
print((two['Total_Effort_Minutes'] / 60 * median_rate).round(0))
print("Total:", round((two['Total_Effort_Minutes'] / 60 * median_rate).sum()))

6845      577429.0
11191    2311203.0
Name: Total_Effort_Minutes, dtype: float64
Total: 2888632


In [ ]:
cl = pd.read_csv('tawos_data/Change_Log.csv',
                 usecols=['Issue_ID', 'Change_Type', 'Field', 'From_String', 'To_String', 'Creation_Date'])
pe = cl[(cl['Change_Type'] == 'PEOPLE') & (cl['Field'] == 'assignee')
        & cl['Issue_ID'].isin(un['ID'])].sort_values('Creation_Date')

last = pe.groupby('Issue_ID').tail(1)                    # last assignee event per issue
last = last.assign(Prev_Name=last['To_String'].fillna(last['From_String']))
print("Unassigned issues:", len(un))
print("With any assignee event:", pe['Issue_ID'].nunique())
print("With a recoverable name:", last['Prev_Name'].notna().sum())
print(last['Prev_Name'].value_counts().head(10))

Unassigned issues: 1867
With any assignee event: 288
With a recoverable name: 288
Prev_Name
164187            69
164408            37
<UNKNOWN_USER>    25
164038            19
164130            18
163693            15
164714            14
165178            13
164609            10
164837             9
Name: count, dtype: int64


In [ ]:
pd.read_csv('tawos_data/User.csv').columns.tolist()

['ID', 'Project_ID']

In [ ]:
last = last.copy()
last['Recovered_ID'] = pd.to_numeric(last['Prev_Name'], errors='coerce')   # <UNKNOWN_USER> -> NaN
rec = last[['Issue_ID', 'Recovered_ID']].dropna()
print("Usable:", len(rec), "| unknown user:", last['Recovered_ID'].isna().sum())

# Who are they? (Core / One-off / Other / not in user_stats)
grp = user_stats.set_index('Assignee_ID')['Group']
print(rec['Recovered_ID'].map(grp).value_counts(dropna=False))
print(rec['Recovered_ID'].value_counts().head(3))

# Did the last event clear the assignee, or set one?
print("Last event cleared it:", last['To_String'].isna().sum(), "| set an assignee:", last['To_String'].notna().sum())

# How much effort is recoverable, and are the two big tickets in it?
eff = un.set_index('ID')['Total_Effort_Minutes']
print("Effort share recoverable:", round(eff.reindex(rec['Issue_ID']).sum() / eff.sum(), 3))
print("Big tickets recovered:", rec['Issue_ID'].isin(two['ID']).sum(), "of", len(two))

Usable: 263 | unknown user: 25
Recovered_ID
Core       251
Other        9
One-off      2
NaN          1
Name: count, dtype: int64
Recovered_ID
164187.0    69
164408.0    37
164038.0    19
Name: count, dtype: int64
Last event cleared it: 288 | set an assignee: 0
Effort share recoverable: 0.539
Big tickets recovered: 1 of 2


In [ ]:
main = main.drop(columns=['Assignee_ID_Recovered'], errors='ignore')
main = main.merge(rec.rename(columns={'Issue_ID': 'ID', 'Recovered_ID': 'Assignee_ID_Recovered'}),
                  on='ID', how='left')

no_owner = main['Assignee_ID'].isna()
rate_lookup = rates.drop_duplicates('Assignee_ID').set_index('Assignee_ID')['Hourly_Rate_USD']

rec_rate = main['Assignee_ID_Recovered'].map(rate_lookup)
use_rec = no_owner & rec_rate.notna()
main.loc[use_rec, 'Hourly_Rate_USD'] = rec_rate[use_rec]
main.loc[use_rec, 'Rate_Source'] = 'Recovered owner (change log)'

rest = no_owner & ~use_rec
main.loc[rest, 'Hourly_Rate_USD'] = median_rate
main.loc[rest, 'Rate_Source'] = 'Assumed (no assignee, core median)'
main['Is_Unassigned'] = no_owner

print(main['Rate_Source'].value_counts())

Rate_Source
Tier-based                            18642
Assumed (no assignee, core median)     1605
Assumed (core median)                   345
Recovered owner (change log)            262
Name: count, dtype: int64


In [ ]:
na = main[main['Rate_Source'] == 'Assumed (no assignee, core median)']
total_hours = main['Total_Effort_Minutes'].sum() / 60

summary = na.groupby('Status').agg(
    Issues=('ID', 'count'),
    Issues_With_Effort=('Total_Effort_Minutes', lambda x: (x > 0).sum()),
    Effort_Hours=('Total_Effort_Minutes', lambda x: round(x.sum() / 60)),
)
summary['Pct_of_All_Effort'] = (summary['Effort_Hours'] / total_hours * 100).round(2)

print(len(na), "issues |", round(na['Total_Effort_Minutes'].sum() / 60), "hours |",
      round(na['Total_Effort_Minutes'].sum() / 60 / total_hours * 100, 1), "% of all effort\n")
print(summary.sort_values('Effort_Hours', ascending=False), "\n")
print(na['Project_Name'].value_counts(), "\n")

# Where does the effort concentrate?
print(na.nlargest(5, 'Total_Effort_Minutes')[['Issue_Key', 'Status', 'Type', 'Total_Effort_Minutes']])
print("Share of this group's effort in its top 5 issues:",
      round(na.nlargest(5, 'Total_Effort_Minutes')['Total_Effort_Minutes'].sum()
            / na['Total_Effort_Minutes'].sum() * 100, 1), "%")

1605 issues | 93134 hours | 3.7 % of all effort

             Issues  Issues_With_Effort  Effort_Hours  Pct_of_All_Effort
Status                                                                  
Closed         1099                  10         74305               2.99
Resolved          5                   1         18395               0.74
Done            243                   4           434               0.02
In Progress       1                   0             0               0.00
New             116                   0             0               0.00
Open            141                   0             0               0.00 

Project_Name
Mule                        1593
Titanium Mobile Platform       8
Appcelerator Studio            4
Name: count, dtype: int64 

         Issue_Key    Status Type  Total_Effort_Minutes
6808    MULE-10161    Closed  Bug             1657321.0
6845    MULE-10113  Resolved  Bug             1103719.0
15982  TISTUD-7442    Closed  Bug              799222.0
6

In [ ]:
print(pd.crosstab(na['Status'], na['Resolution'].fillna('(blank)')))

# Compare with assigned issues, to see whether this group is different
closed_na = na[na['Status'] == 'Closed']['Resolution'].fillna('(blank)').value_counts(normalize=True)
closed_assigned = main[(main['Status'] == 'Closed') & main['Assignee_ID'].notna()]['Resolution'].fillna('(blank)').value_counts(normalize=True)
print((pd.concat([closed_na, closed_assigned], axis=1, keys=['No assignee', 'Assigned']) * 100).round(1))

Resolution   (blank)  Abandoned  Cannot Reproduce  Completed  Done  Duplicate  \
Status                                                                          
Closed             0          0                21          9     8         34   
Done               0          2                 0         12    11         13   
In Progress        1          0                 0          0     0          0   
New              116          0                 0          0     0          0   
Open             141          0                 0          0     0          0   
Resolved           0          0                 0          0     2          1   

Resolution   Fixed  Incomplete  Rejected  Won't Do  Won't Fix or Usage Issue  \
Status                                                                         
Closed          60           7         1       140                       815   
Done            30           2         0        27                       146   
In Progress      0           0 

In [ ]:
grp = main['Rate_Source'] == 'Assumed (no assignee, core median)'
worked = main['Resolution'].isin(['Fixed', 'Completed', 'Done'])

main['Unassigned_Type'] = np.select(
    [grp & worked, grp & ~worked],
    ['Worked, owner unknown', 'Not worked (closed without fix / still open)'],
    default='Has owner')

na = main[grp]
print(na.groupby('Unassigned_Type')['Total_Effort_Minutes'].agg(
    Issues='count', With_Effort=lambda x: (x > 0).sum(), Hours=lambda x: round(x.sum() / 60)))

# The 15 issues that carry any effort
print(na[na['Total_Effort_Minutes'] > 0][['Issue_Key', 'Status', 'Resolution', 'Type', 'Total_Effort_Minutes']]
      .sort_values('Total_Effort_Minutes', ascending=False))

# Was "Won't Fix" a bulk cleanup? Look for closing dates with many issues
wf = na[na['Resolution'] == "Won't Fix or Usage Issue"]
print(wf['Resolution_Date'].dt.to_period('M').value_counts().head(5))
print(wf['Resolution_Date'].dt.date.value_counts().head(5))

                                              Issues  With_Effort  Hours
Unassigned_Type                                                         
Not worked (closed without fix / still open)    1472            6  63076
Worked, owner unknown                            133            9  30058
         Issue_Key    Status                Resolution   Type  \
6808    MULE-10161    Closed                  Won't Do    Bug   
6845    MULE-10113  Resolved  Won't Fix or Usage Issue    Bug   
15982  TISTUD-7442    Closed                     Fixed    Bug   
6746    MULE-10237    Closed          Cannot Reproduce    Bug   
15281  TISTUD-8736    Closed                     Fixed    Bug   
8912     MULE-7415    Closed  Won't Fix or Usage Issue    Bug   
116     TIDOC-3251    Closed                     Fixed    Bug   
14975  TISTUD-9107    Closed                     Fixed    Bug   
14961  TISTUD-9123    Closed                     Fixed    Bug   
9502     MULE-6663      Done                     Fixed  St

## Issues without an assignee

1,867 of the 20,854 issues (9%) have no assignee, and 1,838 of them (98%) are in Mule
(15.6% of that project's issues, against 0.3-0.4% in the other two).

**Recovering owners.** For 263 of them the change log shows who held the issue before the
assignee was cleared, so they were linked to that person. About 1,600 stay unassigned.

**What the remaining issues are.** The closing reason (Resolution) of the 1,605
that could not be recovered:
- Of the 1,099 closed issues, 74% were closed as *Won't Fix or Usage Issue* and another 13% as
  *Won't Do*. Assigned issues are the opposite (73% *Fixed*).
- A few mass closures explain much of it: 284 issues were closed on 2016-04-05 and 165 on
  2016-07-14, with another 89 and 83 on two more dates. These look like backlog cleanups, not work.
- 141 issues are still Open and 116 New, none with recorded effort.

**Decision.** The group is split in two and kept in the data, not dropped:

| Label | Issues | Meaning |
|---|---|---|
| Worked, owner unknown | 133 | Resolved as Fixed, Completed or Done: work was done, nobody was recorded |
| Not worked | 1,472 | Closed without a fix, or still open: mostly housekeeping |

**Effort caveat.** Only 15 of the 1,605 issues have any effort, but they carry 93,134 hours, and
five tickets hold 88% of that. Four of the biggest were rejected as bugs, so that effort almost
certainly measures how long the ticket stayed open, not work done (see the earlier note on
`Total_Effort_Minutes`).

In [ ]:
main['Assignee_ID_Final'] = main['Assignee_ID'].fillna(main['Assignee_ID_Recovered'])
main['Owner_Source'] = np.select(
    [main['Assignee_ID'].notna(), main['Assignee_ID_Recovered'].notna()],
    ['Original', 'Recovered (change log)'], default='None')
print(main['Owner_Source'].value_counts())

# Were the recovered issues actually worked?
rc = main[main['Owner_Source'] == 'Recovered (change log)']
print(rc['Resolution'].fillna('(blank)').value_counts().head(8))
print(rc['Status'].value_counts())
print("With effort:", (rc['Total_Effort_Minutes'] > 0).sum(),
      "| hours:", round(rc['Total_Effort_Minutes'].sum() / 60))

Owner_Source
Original                  18987
None                       1604
Recovered (change log)      263
Name: count, dtype: int64
Resolution
Won't Fix or Usage Issue    115
Fixed                        90
Incomplete                   20
(blank)                      16
Duplicate                    12
Won't Do                      5
Works as Designed             1
Cannot Reproduce              1
Name: count, dtype: int64
Status
Closed         208
Done            37
Open            12
New              3
Resolved         2
In Progress      1
Name: count, dtype: int64
With effort: 48 | hours: 108717


In [ ]:
main['Assignee_ID_Final'] = (main['Assignee_ID']
                             .fillna(main['Assignee_ID_Recovered'])
                             .astype('Int64'))          # nullable int: IDs have gaps

assert team_profiles['Assignee_ID'].is_unique           # one card per person

keep = ['Assignee_ID', 'Group', 'Seniority_Tier', 'Tier_Confidence', 'HRA_Employee_ID',
        'Job_Role', 'Department', 'Experience_Years',
        'Performance_Score', 'Experience_Score', 'Growth_Score', 'Reliability_Score']
tp = team_profiles[keep].rename(columns={'Assignee_ID': 'Assignee_ID_Final'})
tp['Assignee_ID_Final'] = tp['Assignee_ID_Final'].astype('Int64')

print("Columns in both:", (set(tp.columns) & set(main.columns)) - {'Assignee_ID_Final'})  # should be empty

main_full = main.merge(tp, on='Assignee_ID_Final', how='left', validate='m:1')
assert len(main_full) == len(main)                      # no rows gained or lost

print("Issues with a profile:", main_full['Job_Role'].notna().sum(), "of", len(main_full))
print(main_full.groupby('Owner_Source')['Job_Role'].apply(lambda s: s.notna().mean()).round(2))

Columns in both: set()
Issues with a profile: 18895 of 20854
Owner_Source
None                      0.00
Original                  0.98
Recovered (change log)    0.96
Name: Job_Role, dtype: float64


In [ ]:
#Fill nulls unrecovered
main['Assignee_ID_Final'] = main['Assignee_ID_Final'].fillna(-1).astype('int64')


In [ ]:
main.info()

<class 'pandas.DataFrame'>
RangeIndex: 20854 entries, 0 to 20853
Data columns (total 40 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       20854 non-null  int64         
 1   Issue_Key                20854 non-null  str           
 2   Title                    20854 non-null  str           
 3   Description              20102 non-null  str           
 4   Description_Text         20102 non-null  str           
 5   Type                     20854 non-null  str           
 6   Priority                 20205 non-null  str           
 7   Status                   20854 non-null  str           
 8   Resolution               19219 non-null  str           
 9   Creation_Date            20854 non-null  datetime64[us]
 10  Estimation_Date          8873 non-null   datetime64[us]
 11  Resolution_Date          19218 non-null  datetime64[us]
 12  Last_Updated             20854 non-null  da

In [ ]:
rc['Resolution'].value_counts()

Resolution
Won't Fix or Usage Issue    115
Fixed                        90
Incomplete                   20
Duplicate                    12
Won't Do                      5
Works as Designed             1
Cannot Reproduce              1
Invalid                       1
Done                          1
Hold                          1
Name: count, dtype: int64

In [ ]:
main_full['Owner_Category'] = np.select(
    [main_full['Job_Role'].notna(),
     main_full['Assignee_ID_Final'].notna(),
     main_full['Unassigned_Type'] == 'Worked, owner unknown'],
    ['Tiered contributor',
     'Other contributor (not tiered)',
     'Unassigned - worked, owner unknown'],
    default='Unassigned - not worked / still open')

print(main_full['Owner_Category'].value_counts())
assert main_full['Owner_Category'].notna().all()

Owner_Category
Tiered contributor                      18895
Unassigned - not worked / still open     1471
Other contributor (not tiered)            355
Unassigned - worked, owner unknown        133
Name: count, dtype: int64


In [ ]:
x = main_full[(main_full['Owner_Category'] == 'Other contributor (not tiered)')
              & (main_full['Assignee_ID'].isna())]
print(len(x), "unassigned issues with a recovered ID outside the tiered team")
print(x[['Issue_Key', 'Assignee_ID_Recovered', 'Resolution']])

10 unassigned issues with a recovered ID outside the tiered team
        Issue_Key  Assignee_ID_Recovered Resolution
8      TIDOC-3392                 8845.0        NaN
15     TIDOC-3385                 8845.0   Won't Do
6610   MULE-10392               164118.0        NaN
13039   MULE-2311               164608.0      Fixed
13046   MULE-2304               164608.0      Fixed
13132   MULE-2212               164608.0      Fixed
13360   MULE-1945               164608.0      Fixed
13723   MULE-1492               164869.0      Fixed
13808   MULE-1367               164434.0      Fixed
14423    MULE-545               164825.0      Fixed


In [ ]:
main_full.to_csv('tawos_data/main_part3.csv', index=False)
print(main_full.shape)

(20854, 52)


## who owns each issue

Every one of the 20,854 issues now has an **owner category**, so a blank profile always has a reason:

| Owner_Category | Issues | Meaning |
|---|---|---|
| Tiered contributor | 18,895 | Owner is one of the 182 scored contributors, with a seniority tier and an HRA stand-in |
| Other contributor (not tiered) | 355 | Owner is a real person with 2-9 issues (or 10+ within 30 days): too little data for a tier |
| Unassigned - worked, owner unknown | 133 | No assignee, resolved as Fixed, Completed or Done: work was done and nobody was recorded |
| Unassigned - not worked / still open | 1,471 | No assignee, closed without a fix or still open: mostly backlog housekeeping |

**Key columns**
- `Assignee_ID`: the original assignee. The tiers were calculated from this.
- `Assignee_ID_Recovered`: for 263 unassigned issues, the previous assignee found in the change log.
- `Assignee_ID_Final`: original, or recovered if there was none. This is the key that links the issues to `team_profiles`. It stays empty where no owner can be found, instead of using a placeholder.
- `Owner_Source`: Original, Recovered (change log) or None.

**Caveat on recovered owners.** In all 288 cases the assignee was cleared, so the recovered name is who held the issue before that. Of the 263 usable ones, 115 were closed as *Won't Fix* and 90 as *Fixed*, so the recovered owner is not always the person who did the work. Analyses of workload should consider `Owner_Source`.

In [ ]:
for f in ['projects_dataset', 'evm_monthly', 'project_budgets', 'team_profiles']:
    src = f'tawos_data/{f}.csv'
    if os.path.exists(src) and not os.path.exists(f'tawos_data/{f}_v1.csv'):
        os.rename(src, f'tawos_data/{f}_v1.csv')

In [ ]:
pdset = pd.read_csv('tawos_data/projects_dataset.csv')
tp    = pd.read_csv('tawos_data/team_profiles_v1.csv')

print('IDs non-null in CSV:', pdset['Assignee_ID_Final'].notna().sum(), 'of', len(pdset))
print('dtype:', pdset['Assignee_ID_Final'].dtype)
print('team_profiles columns:', [c for c in tp.columns if 'ssign' in c or 'ID' in c or 'id' in c])
print(tp.dtypes.head(8))

k = pdset['Assignee_ID_Final'].dropna().astype('int64')
print('share of IDs found in team_profiles:', k.isin(tp['Assignee_ID']).mean())

IDs non-null in CSV: 20854 of 20854
dtype: int64
team_profiles columns: ['Assignee_ID', 'Tier_Confidence', 'HRA_Employee_ID', 'Employee_ID', 'Tasks_Assigned']
Assignee_ID                 int64
Total_Issues                int64
First_Activity                str
Last_Activity                 str
Avg_Resolution_Minutes    float64
Tenure_Days                 int64
Group                         str
Avg_Story_Point           float64
dtype: object
share of IDs found in team_profiles: 0.9060611873021962
